In [ ]:
#trying to pull snow report data for a physical snow report board

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://opensnow.com/explore/season-passes/ikonpass/snow-summary"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def fetch_snow_summary():
    resp = requests.get(URL, headers=HEADERS)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def parse_summary(soup):
    data = []
    resort_divs = soup.select("div")  # adjust selector based on real HTML structure
    for div in resort_divs:
        # Only parse those with a resort name heading
        name_tag = div.find("h3")
        if not name_tag:
            continue
        name = name_tag.text.strip()

        # Last 24 hours snowfall
        last24 = div.find(string="Last 24 Hours")
        if last24:
            val = last24.find_next().get_text(strip=True)
        else:
            val = None

        # Base depth and temperature might be in "Snow Report" tab
        base = None
        temp = None
        # look for sibling elements or classes like 'Base Depth' / 'Temperature'
        bd = div.find(string="Base Depth")
        if bd:
            base = bd.find_next().get_text(strip=True)
        temp_tag = div.find(string="Temperature")
        if temp_tag:
            temp = temp_tag.find_next().get_text(strip=True)

        data.append({
            "Resort": name,
            "24h Snow (in)": val,
            "Base Depth": base,
            "Temperature": temp
        })

    return pd.DataFrame(data)

if __name__ == "__main__":
    soup = fetch_snow_summary()
    df = parse_summary(soup)
    print(df)
    df.to_csv("ikon_snow_summary.csv", index=False)


                         Resort  \
0    6,220 ft • Bolzano • Italy   
1    6,220 ft • Bolzano • Italy   
2    6,220 ft • Bolzano • Italy   
3    6,220 ft • Bolzano • Italy   
4    6,220 ft • Bolzano • Italy   
..                          ...   
180                          0"   
181                          0"   
182                          0"   
183                          0"   
184                          0"   

                                         24h Snow (in) Base Depth Temperature  
0    Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
1    Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
2    Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
3    Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
4    Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
..                                                 ...        ...         ...  
180  Next 1-5 Days0

In [2]:
import re

# Helper to convert snowfall strings to float inches
def parse_snow(s):
    if not s:
        return 0.0
    s = s.lower().strip()
    if "trace" in s:
        return 0.1  # or 0 depending on how you want to treat trace
    match = re.search(r"(\d+(\.\d+)?)", s)
    return float(match.group(1)) if match else 0.0

# Add numerical snowfall column
df["24h Snow (in) Parsed"] = df["24h Snow (in)"].apply(parse_snow)

# Sort by parsed snowfall in descending order
df_sorted = df.sort_values("24h Snow (in) Parsed", ascending=False)

# Display or save sorted data
print(df_sorted[["Resort", "24h Snow (in)", "Base Depth", "Temperature"]].head(10))  # top 10
df_sorted.to_csv("ikon_snow_summary_sorted.csv", index=False)


                                      Resort  \
0                 6,220 ft • Bolzano • Italy   
167               5,978 ft • Bolzano • Italy   
117                                       0"   
90                                        0"   
32           9,895 ft • Utah • United States   
135                                       0"   
86               7,001 ft • Alberta • Canada   
36                                        0"   
158  1,860 ft • Pennsylvania • United States   
122                997 ft • Ontario • Canada   

                                         24h Snow (in) Base Depth Temperature  
0    Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
167  Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
117  Next 1-5 Days0"Next 6-10 Days0"TTue 22WWed 23T...       None        None  
90   Next 1-5 Days0"Next 6-10 Days0"WWed 23TThu 24F...       None        None  
32   Next 1-5 Days0"Next 6-10 Days0"WWed 23TThu 24F...       None      